# Figure 3: learning during Figure-8 and DNMP T-maze training

Panels **3d, 3e and 3h**. Independent post-training behavior (3i) and schematic assembly are outside this notebook.

This notebook follows **Xu et al.**. Data loading, model fitting, panel selection and plotting are shown below as executable steps. Shared scientific methods remain in `src/`; this notebook does not delegate the figure to `run_figure()`.

Start with the separate manuscript source-data files listed in `data/required_files.csv`. These are analysis source tables and frozen map/embedding arrays, rather than raw movies. The aligned-data computation examples are in `00_recompute_metrics.ipynb`.

Run cells from top to bottom. `REFIT_STATISTICS=True` refits the manuscript models; set it to `False` to use the frozen estimates. Outputs are saved separately from inputs.

## 1. Paths and execution parameters

Set `PPRH_DATA_DIR` or edit `DATA_ROOT`. The default refits statistical models. No raw neural resampling is triggered here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "main.py").is_file() and (p / "src/paper.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, display
from paper_io import read_table, save_panel as _save_panel, write_tables, p_symbol, write_run_record

DATA_ROOT = Path(os.environ.get("PPRH_DATA_DIR", str(ROOT / "data/input")))
OUTPUT_ROOT = Path(os.environ.get("PPRH_OUTPUT_DIR", str(ROOT / "outputs/notebooks")))
REFIT_STATISTICS = os.environ.get("PPRH_REFIT_STATISTICS", "1") == "1"
FORMATS = ("svg", "png")
DPI = 300

data_root = DATA_ROOT
output = OUTPUT_ROOT / "Fig3"
refit, formats, dpi = REFIT_STATISTICS, FORMATS, DPI
output.mkdir(parents=True, exist_ok=True)
if not data_root.is_dir():
    raise FileNotFoundError("Supply the separate manuscript inputs through DATA_ROOT; see data/README.md.")


## 2. Analysis contract

Daily LMMs retain mouse and D1–D5, with group, centered day, group × day, standardized mean speed and route-relative lateral deviation. The reference group is C and the comparison is I. Fig.3e uses binary trials 1–90 in a mouse-clustered binomial GEE, exchangeable working correlation, bias-reduced covariance and Student-t inference (`n_mouse - 4` degrees of freedom, 8 in the frozen cohort). Moving windows are display summaries. Group × day and signed C/I slope/end-point contrasts are retained; no across-day mouse averaging is added.

## 3. Notebook functions

The export function saves and previews each panel. Where needed, the source-table loader or annotation helper is defined here in full. Numerical estimators and shared plotting functions are imported in the relevant analysis cells.

In [ ]:
def save_panel(fig, output_root, panel, description, **kwargs):
    """Save one panel with the shared export method, then preview its result."""
    paths = _save_panel(fig, output_root, panel, description, **kwargs)
    svg = next((p for p in paths if p.endswith(".svg")), None)
    png = next((p for p in paths if p.endswith(".png")), None)
    if svg:
        display(SVG(filename=svg))
    elif png:
        display(Image(filename=png))
    return paths


In [ ]:
def _annotations(frame):
    if 'model_variant' in frame:
        frame = frame[frame.model_variant.eq('primary_adjusted_LMM')]
    if 'model_role' in frame:
        frame = frame[frame.model_role.eq('adjusted')]

    def symbol(name):
        part = frame[frame.contrast.eq(name)]
        return p_symbol(part.iloc[0].p_value) if len(part) == 1 else None
    return dict(overall_annotation=symbol('centered_overall_group_effect'), slope_annotations={'C': symbol('reference_group_day_slope'), 'I': symbol('comparison_group_day_slope')}, interaction_annotation=symbol('group_by_day_interaction'))


## 4. Source data, statistics and panel-by-panel plots

Each panel cell names its source, contrast and display parameters. Use the current panel names below; historical filenames remain unchanged for traceability. Model fitting can take longer than figure rendering.

### Shared methods used below

In [ ]:
from figure_helpers import figure3 as h


### Shared methods used below

In [ ]:
from figure_helpers import figure3_trial_gee as gee


### Prepare the next analysis step

In [ ]:
from maze_analysis.manuscript_lmm import fit_adjusted_unadjusted_lmm

exports = []


### Prepare the next analysis step

In [ ]:
panel, prefix, plotter = ('d', '01_Fig3D_Figure8', h.plot_fig3d)


### Read the source data

In [ ]:
observations = read_table(data_root, f'fig3/source_data/{prefix}_success_mouse_day.csv')

summary = read_table(data_root, f'fig3/source_data/{prefix}_success_group_summary.csv')

statistics = read_table(data_root, f'fig3/stats/{prefix}_success_LMM_estimands.csv')


In [ ]:
display(observations.head())


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    fit = fit_adjusted_unadjusted_lmm(observations, value_col='success_rate', time_col='day', group_col='group', mouse_col='mouse', reference_group='C', comparison_group='I', continuous_covariates=('mean_speed_z', 'route_relative_lateral_deviation_z'), first_time=1, last_time=5)['adjusted']
    if not fit.succeeded:
        raise RuntimeError(fit.diagnostics)
    statistics = fit.contrasts
    write_tables({f'Fig3{panel}_refitted_estimands': statistics}, Path(output) / 'statistics')


In [ ]:
display(statistics.head())


### Fig3 d — draw and export

In [ ]:
fig = plotter(observations, summary, **_annotations(statistics))[0]

exports += save_panel(fig, output, f'Fig3_{panel}', 'learning_curve', formats=formats, dpi=dpi)


### Prepare the next analysis step

In [ ]:
panel, prefix, plotter = ('h', '03_Fig3H_Tmaze', h.plot_fig3h)


### Read the source data

In [ ]:
observations = read_table(data_root, f'fig3/source_data/{prefix}_success_mouse_day.csv')

summary = read_table(data_root, f'fig3/source_data/{prefix}_success_group_summary.csv')

statistics = read_table(data_root, f'fig3/stats/{prefix}_success_LMM_estimands.csv')


In [ ]:
display(observations.head())


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    fit = fit_adjusted_unadjusted_lmm(observations, value_col='success_rate', time_col='day', group_col='group', mouse_col='mouse', reference_group='C', comparison_group='I', continuous_covariates=('mean_speed_z', 'route_relative_lateral_deviation_z'), first_time=1, last_time=5)['adjusted']
    if not fit.succeeded:
        raise RuntimeError(fit.diagnostics)
    statistics = fit.contrasts
    write_tables({f'Fig3{panel}_refitted_estimands': statistics}, Path(output) / 'statistics')


In [ ]:
display(statistics.head())


### Fig3 h — draw and export

In [ ]:
fig = plotter(observations, summary, **_annotations(statistics))[0]

exports += save_panel(fig, output, f'Fig3_{panel}', 'learning_curve', formats=formats, dpi=dpi)


### Fig3 e — prepare panel inputs

In [ ]:
individual = read_table(data_root, 'fig3/source_data/02_Fig3E_Figure8_moving_success_mouse_trial_display_common_support.csv')

summary = read_table(data_root, 'fig3/source_data/02_Fig3E_Figure8_moving_success_group_summary_common_support.csv')


### Fig3 e — draw and export

In [ ]:
fig = h.plot_fig3e(individual, summary, common_support_max=90)[0]

exports += save_panel(fig, output, 'Fig3_e', 'trial_resolved_learning', formats=formats, dpi=dpi)


### Fit the manuscript models or retain the frozen estimates

In [ ]:
if refit:
    trials, _ = gee.prepare_data(Path(data_root) / 'fig3_trial/atomic.csv')
    _, params, covariance = gee.fit_gee(trials)
    estimates = gee.all_estimands(params, covariance, df=trials.mouse.nunique() - 4)
    write_tables({'Fig3e_refitted_GEE': estimates}, Path(output) / 'statistics')


## 5. Export record

Panel files and any refitted model tables are saved under the figure output directory. The record identifies the selected manuscript workflow and whether statistics were refitted.

In [ ]:
write_run_record(output / "run.json", figure="3", refit=refit, exports=exports)
display(pd.DataFrame({"exported_file": [str(Path(p).relative_to(output)) for p in exports]}))
